# 11a — Cross-Validation Pipeline & Feature Selection

This notebook implements a reproducible **5-fold StratifiedGroupKFold** cross-validation and feature-selection workflow for the integrated datasets produced during data integration.

## Modeling datasets

The cross-validation pipeline is run independently on:

- `demographics_questionnaire.csv`
- `wearable_questionnaire.csv`
- `multimodal_full.csv`

These are the participant-level modeling datasets produced from the source data. The original `patients.csv`, `questionnaire.csv`, and `movement_metadata.csv` files are documented for traceability but are not passed directly into this cross-validation pipeline because the integrated datasets already contain the aligned participant-level features required for modeling.

## Objective

For each integrated dataset, the notebook:

- uses `patient_id` as the grouping variable in `StratifiedGroupKFold`;
- performs preprocessing inside each training fold;
- performs feature selection inside each training fold;
- applies the fitted transformations and selected features to the held-out validation fold;
- evaluates fold-level performance using class-weighted multinomial Logistic Regression;
- exports selected feature sets and selected train/validation datasets;
- produces feature-selection stability and fold-performance summaries;
- verifies that validation information is not used during preprocessing or feature selection.

`N_SELECTED_FEATURES = 20` is used as an initial fixed value for consistent stability analysis. It is not treated as the final optimal value and should be optimized during hyperparameter tuning. The selected features generated here are primarily used for stability analysis; feature selection should be re-fit inside the later tuning pipeline.

## 1. Libraries and project paths

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 200)

cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the src directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for directory in [METRICS_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"Outputs: {OUTPUTS_DIR}")

## 2. Dataset inventory

In [ ]:
# Source files used during integration.
SOURCE_DATA_FILES = {
    "patients": DATA_DIR / "raw" / "patients.csv",
    "questionnaire": DATA_DIR / "raw" / "questionnaire.csv",
    "movement_metadata": DATA_DIR / "raw" / "movement_metadata.csv",
}

# Participant-level modeling datasets produced by the integration stage.
MODELING_DATA_FILES = {
    "demographics_questionnaire": (
        PROCESSED_DIR / "demographics_questionnaire.csv"
    ),
    "wearable_questionnaire": (
        PROCESSED_DIR / "wearable_questionnaire.csv"
    ),
    "multimodal_full": (
        PROCESSED_DIR / "multimodal_full.csv"
    ),
}

dataset_inventory = []

for dataset_name, dataset_path in {
    **SOURCE_DATA_FILES,
    **MODELING_DATA_FILES,
}.items():
    dataset_inventory.append({
        "dataset": dataset_name,
        "path": str(dataset_path),
        "exists": dataset_path.exists(),
        "role": (
            "modeling"
            if dataset_name in MODELING_DATA_FILES
            else "source"
        ),
    })

dataset_inventory = pd.DataFrame(dataset_inventory)
display(dataset_inventory)

The three integrated modeling datasets are evaluated independently. The source files are retained in the notebook inventory to document how the modeling datasets originate, but they are not re-merged or modeled directly here.

## 3. Cross-validation and feature-selection configuration

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
N_SELECTED_FEATURES = 20

TARGET_COLUMN = "label"
PARTICIPANT_ID_COLUMN = "patient_id"

EXCLUDED_COLUMNS = [
    PARTICIPANT_ID_COLUMN,
    TARGET_COLUMN,
    "condition_group",
    "condition_original",
]

print(f"Cross-validation strategy: StratifiedGroupKFold")
print(f"Cross-validation folds: {N_SPLITS}")
print(f"Grouping variable: {PARTICIPANT_ID_COLUMN}")
print(f"Initial selected features per fold: {N_SELECTED_FEATURES}")
print(f"Random state: {RANDOM_STATE}")

`N_SELECTED_FEATURES = 20` is used as a fixed starting point so that feature-selection stability can be compared consistently across folds and datasets. This value is not assumed to be optimal. The number of selected features should be included in the later hyperparameter-tuning search.

`StratifiedGroupKFold` is used to explicitly keep records from the same `patient_id` within a single fold while preserving class balance as closely as possible. The current integrated datasets contain one row per participant, but using the grouped strategy aligns the pipeline with the agreed validation approach and keeps it robust if repeated participant observations are introduced later.

## 4. Load and validate all modeling datasets

In [ ]:
datasets = {}
dataset_structure_records = []

for dataset_name, dataset_path in MODELING_DATA_FILES.items():
    assert dataset_path.exists(), (
        f"Required modeling dataset not found: {dataset_path}"
    )

    df = pd.read_csv(
        dataset_path,
        dtype={PARTICIPANT_ID_COLUMN: str},
    )

    required_columns = {
        PARTICIPANT_ID_COLUMN,
        TARGET_COLUMN,
    }

    missing_required = required_columns.difference(df.columns)

    assert not missing_required, (
        f"{dataset_name} is missing required columns: "
        f"{sorted(missing_required)}"
    )

    assert df[PARTICIPANT_ID_COLUMN].notna().all()
    assert df[TARGET_COLUMN].notna().all()

    datasets[dataset_name] = df

    participant_counts = (
        df[PARTICIPANT_ID_COLUMN]
        .value_counts()
    )

    dataset_structure_records.append({
        "dataset": dataset_name,
        "rows": len(df),
        "columns": df.shape[1],
        "unique_participants": (
            df[PARTICIPANT_ID_COLUMN].nunique()
        ),
        "duplicate_patient_rows": (
            df[PARTICIPANT_ID_COLUMN]
            .duplicated()
            .sum()
        ),
        "max_rows_per_participant": (
            participant_counts.max()
        ),
        "n_classes": df[TARGET_COLUMN].nunique(),
    })

dataset_structure_summary = pd.DataFrame(
    dataset_structure_records
)

display(dataset_structure_summary)

print("All integrated modeling datasets loaded successfully.")

## 5. Class distributions

In [ ]:
class_distribution_records = []

for dataset_name, df in datasets.items():
    counts = (
        df[TARGET_COLUMN]
        .value_counts()
        .sort_index()
    )

    for label, count in counts.items():
        class_distribution_records.append({
            "dataset": dataset_name,
            "label": label,
            "count": count,
            "percentage": count / len(df) * 100,
        })

class_distribution_summary = pd.DataFrame(
    class_distribution_records
)

display(class_distribution_summary)

class_distribution_summary.to_csv(
    TABLES_DIR / "cv_all_datasets_class_distribution.csv",
    index=False,
)

## 6. Preprocessing helper

In [ ]:
def build_preprocessor(
    numerical_columns,
    categorical_columns,
):
    """Create a fresh preprocessing transformer for one fold."""

    transformers = []

    if numerical_columns:
        numerical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]
        )

        transformers.append(
            (
                "numerical",
                numerical_pipeline,
                numerical_columns,
            )
        )

    if categorical_columns:
        categorical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

A new preprocessor is created for every fold. Numerical imputation, categorical imputation, scaling, encoding, variance filtering, and feature selection are therefore fitted only on that fold's training subset.

## 7. Cross-validation function

In [ ]:
def run_dataset_cross_validation(
    dataset_name,
    df,
):
    """Run grouped 5-fold CV and fold-specific feature selection."""

    feature_columns = [
        column
        for column in df.columns
        if column not in EXCLUDED_COLUMNS
    ]

    X = df[feature_columns].copy()
    y = df[TARGET_COLUMN].copy()
    groups = df[PARTICIPANT_ID_COLUMN].copy()

    numerical_features = (
        X.select_dtypes(include=["number"])
        .columns
        .tolist()
    )

    categorical_features = (
        X.select_dtypes(exclude=["number"])
        .columns
        .tolist()
    )

    cv = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    fold_metrics = []
    selected_feature_records = []
    fold_audit_records = []
    oof_prediction_records = []

    for fold_number, (
        train_index,
        validation_index,
    ) in enumerate(
        cv.split(
            X,
            y,
            groups=groups,
        ),
        start=1,
    ):
        X_train = X.iloc[train_index].copy()
        X_validation = X.iloc[validation_index].copy()

        y_train = y.iloc[train_index].copy()
        y_validation = y.iloc[validation_index].copy()

        train_ids = groups.iloc[train_index].copy()
        validation_ids = groups.iloc[validation_index].copy()

        participant_overlap = (
            set(train_ids)
            .intersection(set(validation_ids))
        )

        assert not participant_overlap, (
            f"{dataset_name}, fold {fold_number}: "
            "participant overlap detected."
        )

        # ------------------------------------------------------
        # Fit preprocessing on training fold only
        # ------------------------------------------------------
        preprocessor = build_preprocessor(
            numerical_features,
            categorical_features,
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            X_train_preprocessed = (
                preprocessor.fit_transform(X_train)
            )

            X_validation_preprocessed = (
                preprocessor.transform(X_validation)
            )

        transformed_feature_names = (
            preprocessor.get_feature_names_out()
        )

        # ------------------------------------------------------
        # Fit zero-variance filter on training fold only
        # ------------------------------------------------------
        variance_filter = VarianceThreshold(
            threshold=0.0
        )

        X_train_variance = variance_filter.fit_transform(
            X_train_preprocessed
        )

        X_validation_variance = variance_filter.transform(
            X_validation_preprocessed
        )

        variance_feature_names = (
            transformed_feature_names[
                variance_filter.get_support()
            ]
        )

        # Use up to 20 features if fewer survive preprocessing.
        k_features = min(
            N_SELECTED_FEATURES,
            X_train_variance.shape[1],
        )

        # ------------------------------------------------------
        # Fit feature selector on training fold only
        # ------------------------------------------------------
        selector = SelectKBest(
            score_func=f_classif,
            k=k_features,
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            X_train_selected = selector.fit_transform(
                X_train_variance,
                y_train,
            )

            X_validation_selected = selector.transform(
                X_validation_variance
            )

        selected_feature_names = (
            variance_feature_names[
                selector.get_support()
            ]
        )

        selected_scores = (
            selector.scores_[
                selector.get_support()
            ]
        )

        # ------------------------------------------------------
        # Train evaluation model
        # ------------------------------------------------------
        classifier = LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )

        classifier.fit(
            X_train_selected,
            y_train,
        )

        validation_predictions = classifier.predict(
            X_validation_selected
        )

        # ------------------------------------------------------
        # Metrics
        # ------------------------------------------------------
        fold_metrics.append({
            "dataset": dataset_name,
            "fold": fold_number,
            "train_participants": len(train_index),
            "validation_participants": len(validation_index),
            "selected_feature_count": k_features,
            "accuracy": accuracy_score(
                y_validation,
                validation_predictions,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_validation,
                validation_predictions,
            ),
            "macro_f1": f1_score(
                y_validation,
                validation_predictions,
                average="macro",
                zero_division=0,
            ),
            "precision_macro": precision_score(
                y_validation,
                validation_predictions,
                average="macro",
                zero_division=0,
            ),
            "recall_macro": recall_score(
                y_validation,
                validation_predictions,
                average="macro",
                zero_division=0,
            ),
        })

        # ------------------------------------------------------
        # Selected features
        # ------------------------------------------------------
        fold_selected = pd.DataFrame({
            "dataset": dataset_name,
            "fold": fold_number,
            "feature": selected_feature_names,
            "selection_score": selected_scores,
        }).sort_values(
            "selection_score",
            ascending=False,
        )

        fold_selected.to_csv(
            TABLES_DIR
            / (
                f"{dataset_name}_cv_fold_{fold_number}"
                "_selected_features.csv"
            ),
            index=False,
        )

        selected_feature_records.extend(
            fold_selected.to_dict("records")
        )

        # ------------------------------------------------------
        # Export selected train/validation datasets
        # ------------------------------------------------------
        selected_train_df = pd.DataFrame(
            X_train_selected,
            columns=selected_feature_names,
        )

        selected_train_df.insert(
            0,
            TARGET_COLUMN,
            y_train.reset_index(drop=True),
        )

        selected_train_df.insert(
            0,
            PARTICIPANT_ID_COLUMN,
            train_ids.reset_index(drop=True),
        )

        selected_validation_df = pd.DataFrame(
            X_validation_selected,
            columns=selected_feature_names,
        )

        selected_validation_df.insert(
            0,
            TARGET_COLUMN,
            y_validation.reset_index(drop=True),
        )

        selected_validation_df.insert(
            0,
            PARTICIPANT_ID_COLUMN,
            validation_ids.reset_index(drop=True),
        )

        selected_train_df.to_csv(
            TABLES_DIR
            / (
                f"{dataset_name}_cv_fold_{fold_number}"
                "_selected_train_dataset.csv"
            ),
            index=False,
        )

        selected_validation_df.to_csv(
            TABLES_DIR
            / (
                f"{dataset_name}_cv_fold_{fold_number}"
                "_selected_validation_dataset.csv"
            ),
            index=False,
        )

        # ------------------------------------------------------
        # Audit
        # ------------------------------------------------------
        fold_audit_records.append({
            "dataset": dataset_name,
            "fold": fold_number,
            "participant_overlap_count": len(
                participant_overlap
            ),
            "preprocessing_fit_scope": "training_fold_only",
            "variance_filter_fit_scope": "training_fold_only",
            "feature_selection_fit_scope": "training_fold_only",
            "selected_feature_count": k_features,
            "validation_used_for_selection": False,
            "check": "PASS",
        })

        oof_prediction_records.extend(
            pd.DataFrame({
                "dataset": dataset_name,
                PARTICIPANT_ID_COLUMN:
                    validation_ids.reset_index(drop=True),
                "fold": fold_number,
                "y_true":
                    y_validation.reset_index(drop=True),
                "y_pred":
                    validation_predictions,
            }).to_dict("records")
        )

        print(
            f"{dataset_name} | Fold {fold_number}: "
            f"macro_f1="
            f"{fold_metrics[-1]['macro_f1']:.3f}, "
            f"selected={k_features}"
        )

    return {
        "fold_performance": pd.DataFrame(
            fold_metrics
        ),
        "selected_features": pd.DataFrame(
            selected_feature_records
        ),
        "audit": pd.DataFrame(
            fold_audit_records
        ),
        "oof_predictions": pd.DataFrame(
            oof_prediction_records
        ),
    }

## 8. Run cross-validation for all integrated datasets

In [ ]:
all_results = {}

for dataset_name, df in datasets.items():
    print("=" * 80)
    print(dataset_name.upper())
    print("=" * 80)

    all_results[dataset_name] = (
        run_dataset_cross_validation(
            dataset_name,
            df,
        )
    )

    print()

## 9. Fold-level performance

In [ ]:
all_fold_performance = pd.concat(
    [
        result["fold_performance"]
        for result in all_results.values()
    ],
    ignore_index=True,
)

display(all_fold_performance)

all_fold_performance.to_csv(
    METRICS_DIR
    / "cv_all_datasets_fold_performance.csv",
    index=False,
)

for dataset_name, result in all_results.items():
    result["fold_performance"].to_csv(
        METRICS_DIR
        / f"{dataset_name}_cv_fold_performance.csv",
        index=False,
    )

## 10. Cross-validation performance summary

In [ ]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "precision_macro",
    "recall_macro",
]

summary_records = []

for dataset_name, result in all_results.items():
    fold_df = result["fold_performance"]

    for metric in metric_columns:
        summary_records.append({
            "dataset": dataset_name,
            "metric": metric,
            "mean": fold_df[metric].mean(),
            "std": fold_df[metric].std(ddof=1),
            "min": fold_df[metric].min(),
            "max": fold_df[metric].max(),
        })

cv_performance_summary = pd.DataFrame(
    summary_records
)

display(cv_performance_summary)

cv_performance_summary.to_csv(
    METRICS_DIR
    / "cv_all_datasets_performance_summary.csv",
    index=False,
)

## 11. Feature-selection summaries

In [ ]:
all_feature_summaries = []

for dataset_name, result in all_results.items():
    selected = result["selected_features"]

    feature_summary = (
        selected
        .groupby("feature")
        .agg(
            times_selected=("fold", "nunique"),
            mean_selection_score=(
                "selection_score",
                "mean",
            ),
            std_selection_score=(
                "selection_score",
                "std",
            ),
        )
        .reset_index()
    )

    feature_summary.insert(
        0,
        "dataset",
        dataset_name,
    )

    feature_summary[
        "selection_percentage"
    ] = (
        feature_summary["times_selected"]
        / N_SPLITS
        * 100
    )

    feature_summary = feature_summary.sort_values(
        [
            "times_selected",
            "mean_selection_score",
        ],
        ascending=[False, False],
    )

    feature_summary.to_csv(
        TABLES_DIR
        / f"{dataset_name}_cv_feature_selection_summary.csv",
        index=False,
    )

    all_feature_summaries.append(
        feature_summary
    )

all_feature_selection_summary = pd.concat(
    all_feature_summaries,
    ignore_index=True,
)

display(all_feature_selection_summary)

all_feature_selection_summary.to_csv(
    TABLES_DIR
    / "cv_all_datasets_feature_selection_summary.csv",
    index=False,
)

The selected features in this notebook are used to examine **selection stability across folds**. They are not intended to become one fixed feature set for the next tuning notebook. During hyperparameter tuning, preprocessing and feature selection should remain inside the tuning pipeline and be re-fit using the training data available in each tuning fold.

## 12. Validation-information checks

In [ ]:
all_audits = pd.concat(
    [
        result["audit"]
        for result in all_results.values()
    ],
    ignore_index=True,
)

display(all_audits)

assert (
    all_audits["participant_overlap_count"]
    .eq(0)
    .all()
)

assert (
    all_audits["validation_used_for_selection"]
    .eq(False)
    .all()
)

assert (
    all_audits["check"]
    .eq("PASS")
    .all()
)

all_audits.to_csv(
    TABLES_DIR
    / "cv_all_datasets_feature_selection_audit.csv",
    index=False,
)

print(
    "Validation-information check: PASS — "
    "preprocessing and feature selection are fitted "
    "using training-fold data only for every dataset."
)

## 13. Out-of-fold performance

In [ ]:
oof_metric_records = []
all_oof_predictions = []

for dataset_name, result in all_results.items():
    oof = result["oof_predictions"].copy()

    all_oof_predictions.append(oof)

    oof_metric_records.append({
        "dataset": dataset_name,
        "accuracy": accuracy_score(
            oof["y_true"],
            oof["y_pred"],
        ),
        "balanced_accuracy": balanced_accuracy_score(
            oof["y_true"],
            oof["y_pred"],
        ),
        "macro_f1": f1_score(
            oof["y_true"],
            oof["y_pred"],
            average="macro",
            zero_division=0,
        ),
        "precision_macro": precision_score(
            oof["y_true"],
            oof["y_pred"],
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            oof["y_true"],
            oof["y_pred"],
            average="macro",
            zero_division=0,
        ),
    })

all_oof_predictions = pd.concat(
    all_oof_predictions,
    ignore_index=True,
)

oof_metrics = pd.DataFrame(
    oof_metric_records
)

display(oof_metrics)

all_oof_predictions.to_csv(
    TABLES_DIR
    / "cv_all_datasets_out_of_fold_predictions.csv",
    index=False,
)

oof_metrics.to_csv(
    METRICS_DIR
    / "cv_all_datasets_out_of_fold_metrics.csv",
    index=False,
)

## 14. Feature-selection stability figures

In [ ]:
for dataset_name in MODELING_DATA_FILES:
    feature_summary = (
        all_feature_selection_summary[
            all_feature_selection_summary["dataset"]
            == dataset_name
        ]
        .head(20)
        .sort_values(
            [
                "times_selected",
                "mean_selection_score",
            ],
            ascending=[True, True],
        )
    )

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    ax.barh(
        feature_summary["feature"],
        feature_summary["selection_percentage"],
    )

    ax.set_xlabel(
        "Selection frequency across folds (%)"
    )
    ax.set_ylabel("Feature")
    ax.set_title(
        f"Feature Selection Stability: "
        f"{dataset_name}"
    )
    ax.set_xlim(0, 100)

    fig.tight_layout()

    figure_path = (
        FIGURES_DIR
        / f"{dataset_name}_cv_feature_selection_stability.png"
    )

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

    print(f"Saved: {figure_path}")

## 15. Fold Macro F1 comparison

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 6)
)

for dataset_name in MODELING_DATA_FILES:
    fold_df = all_fold_performance[
        all_fold_performance["dataset"]
        == dataset_name
    ]

    ax.plot(
        fold_df["fold"],
        fold_df["macro_f1"],
        marker="o",
        label=dataset_name,
    )

ax.set_xlabel("Cross-validation fold")
ax.set_ylabel("Macro F1")
ax.set_title(
    "Macro F1 Across Stratified Grouped CV Folds"
)
ax.set_xticks(range(1, N_SPLITS + 1))
ax.legend()

fig.tight_layout()

figure_path = (
    FIGURES_DIR
    / "cv_all_datasets_fold_macro_f1.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Saved: {figure_path}")

## 16. Out-of-fold confusion matrices

In [ ]:
for dataset_name in MODELING_DATA_FILES:
    oof = all_oof_predictions[
        all_oof_predictions["dataset"]
        == dataset_name
    ]

    labels = sorted(
        oof["y_true"].unique().tolist()
    )

    cm = confusion_matrix(
        oof["y_true"],
        oof["y_pred"],
        labels=labels,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"True_{label}"
            for label in labels
        ],
        columns=[
            f"Pred_{label}"
            for label in labels
        ],
    )

    cm_df.to_csv(
        METRICS_DIR
        / f"{dataset_name}_cv_out_of_fold_confusion_matrix.csv"
    )

    display(cm_df)

    fig, ax = plt.subplots(
        figsize=(6, 5)
    )

    image = ax.imshow(cm)
    fig.colorbar(image, ax=ax)

    ax.set_title(
        f"Out-of-Fold Confusion Matrix: "
        f"{dataset_name}"
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center",
            )

    fig.tight_layout()

    figure_path = (
        FIGURES_DIR
        / (
            f"{dataset_name}"
            "_cv_out_of_fold_confusion_matrix.png"
        )
    )

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

## 17. Deliverables verification

In [ ]:
deliverables = pd.DataFrame({
    "deliverable": [
        "StratifiedGroupKFold cross-validation pipeline",
        "Cross-validation run for demographics + questionnaire",
        "Cross-validation run for wearable + questionnaire",
        "Cross-validation run for full multimodal data",
        "Selected feature lists for every fold",
        "Selected training feature datasets",
        "Selected validation feature datasets",
        "Feature-selection summaries",
        "Fold-performance summaries",
        "Validation-information verification",
        "Out-of-fold metrics",
        "Figures and confusion matrices",
    ],
    "status": ["READY"] * 12,
})

deliverables

## 18. Initial observations

In [ ]:
dataset_mean_performance = (
    all_fold_performance
    .groupby("dataset")[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "precision_macro",
            "recall_macro",
        ]
    ]
    .mean()
    .sort_values(
        "macro_f1",
        ascending=False,
    )
)

display(dataset_mean_performance)

best_dataset = (
    dataset_mean_performance
    .index[0]
)

print(
    "Highest mean Macro F1 dataset: "
    f"{best_dataset}"
)

print(
    "\nInterpret these results together with the "
    "feature-selection stability summaries. "
    "The selected feature lists are exploratory stability "
    "outputs and should be re-estimated during tuning."
)

## 19. Conclusion

This notebook applies the same five-fold `StratifiedGroupKFold` cross-validation and fold-specific feature-selection procedure independently to all three integrated modeling datasets produced during data integration: **demographics + questionnaire**, **wearable + questionnaire**, and **full multimodal**.

For every dataset and fold, preprocessing parameters, zero-variance filtering, ANOVA feature scores, and the selected feature set are learned using the training fold only. `patient_id` is supplied as the grouping variable, ensuring participant-level separation while maintaining class balance as closely as possible.

Twenty features are retained as an initial fixed value to support consistent stability comparisons. This value should be optimized during subsequent hyperparameter tuning. Similarly, the fold-specific feature sets generated here are intended for stability analysis rather than direct reuse as a fixed tuning feature set; feature selection should be re-fit inside the later tuning pipeline.

The generated fold-specific feature datasets, selection summaries, performance summaries, out-of-fold metrics, confusion matrices, and figures provide the required outputs for downstream model optimization and comparison across the integrated data modalities.